# 03 — Exploratory Pricing Analysis

## Plate Value Intelligence

This notebook investigates how licence-plate structure, numeric rarity, repetition, format, and auction-event context relate to observed hammer prices.

### Objectives

- Review the engineered feature table
- Understand the overall price distribution
- Compare price behaviour across auction events
- Test whether short plates, low numbers, repetition, and format groups are associated with higher prices
- Identify commercially meaningful pricing signals
- Prepare evidence for later valuation modelling

### Analytical principle

This notebook is exploratory. Relationships observed here are associations, not proof of causation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

In [2]:
PROJECT_ROOT = Path.cwd()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

FEATURE_FILE = (
    PROCESSED_DATA_DIR
    / "plate_features_2025.csv"
)

print(f"Feature file: {FEATURE_FILE}")
print(f"File exists: {FEATURE_FILE.exists()}")

Feature file: /Users/osmanorka/Plate-Value-Intelligence/data/processed/plate_features_2025.csv
File exists: True


In [3]:
df = pd.read_csv(FEATURE_FILE)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Rows: 17,782
Columns: 75


## 1. Dataset Validation

Critical identifiers, targets, engineered features, and validation-group fields are checked before analysis.

In [4]:
required_columns = [
    "event_code",
    "lot_number",
    "plate_raw",
    "hammer_price_gbp",
    "log_hammer_price",
    "plate_length",
    "numeric_value",
    "is_number_below_100",
    "has_repeated_digit",
    "has_repeated_letter",
    "plate_format_group",
    "low_number_score",
    "validation_group",
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

print(
    f"Missing required columns: "
    f"{missing_required_columns}"
)


Missing required columns: []


In [5]:
assert len(df) == 17_782, "Unexpected row count."
assert df["hammer_price_gbp"].notna().all(), (
    "Missing target prices detected."
)
assert df["hammer_price_gbp"].gt(0).all(), (
    "Non-positive prices detected."
)
assert not missing_required_columns, (
    "Required analysis columns are missing."
)
assert df["validation_group"].eq(df["event_code"]).all(), (
    "Expected validation_group to preserve event-level grouping."
)

print("All exploratory-analysis checks passed.")


All exploratory-analysis checks passed.


## 2. Overall Hammer-Price Profile

The overall price distribution is summarised before comparing individual plate characteristics.

In [6]:
price_summary = (
    df["hammer_price_gbp"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .to_frame(name="hammer_price_gbp")
)

price_summary

Out[0]: 
       hammer_price_gbp
count         17,782.00
mean           2,495.71
std            3,373.40
min              200.00
1%               200.00
5%               230.00
10%              320.00
25%              730.00
50%            1,510.00
75%            3,110.00
90%            5,690.00
95%            7,849.00
99%           14,020.00
max          102,010.00


In [7]:
ax = df["hammer_price_gbp"].plot(
    kind="hist",
    bins=60,
    figsize=(10, 5)
)

ax.set_title("Distribution of Hammer Prices")
ax.set_xlabel("Hammer Price (£)")
ax.set_ylabel("Number of Plates")

plt.tight_layout()
plt.show()

Fallback to a different backend
<ipython-input-1-a8a34b207984>:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
overall_median_price = df["hammer_price_gbp"].median()
baseline_premium_top_10_rate_pct = (
    df["is_premium_top_10pct"].mean()
    * 100
)
minimum_reliable_group_size = 30


def add_pricing_diagnostics(
    summary: pd.DataFrame,
    median_column: str = "median_price",
    count_column: str = "plate_count",
    premium_rate_column: str = "premium_top_10_rate_pct",
) -> pd.DataFrame:
    summary = summary.copy()

    if median_column in summary.columns:
        summary["median_lift_vs_overall"] = (
            summary[median_column]
            / overall_median_price
        )

    if premium_rate_column in summary.columns:
        summary["premium_top_10_lift_vs_baseline"] = (
            summary[premium_rate_column]
            / baseline_premium_top_10_rate_pct
        )

    if count_column in summary.columns:
        summary["sample_size_flag"] = np.where(
            summary[count_column] < minimum_reliable_group_size,
            "small_sample",
            "sufficient_sample",
        )

    return summary


print(f"Overall median hammer price: £{overall_median_price:,.0f}")
print(
    f"Baseline top-10 premium rate: "
    f"{baseline_premium_top_10_rate_pct:,.1f}%"
)
print(f"Small-sample threshold: fewer than {minimum_reliable_group_size} rows")


Overall median hammer price: £1,510
Baseline top-10 premium rate: 10.0%
Small-sample threshold: fewer than 30 rows


In [9]:
ax = df["log_hammer_price"].plot(
    kind="hist",
    bins=60,
    figsize=(10, 5)
)

ax.set_title("Distribution of Log-Transformed Hammer Prices")
ax.set_xlabel("log(1 + Hammer Price)")
ax.set_ylabel("Number of Plates")

plt.tight_layout()
plt.show()

<ipython-input-1-b067fde5f79e>:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Auction Event Pricing Profile

Price levels are compared across auction events to identify event-level shifts and support later time-aware validation.

In [10]:
event_price_summary = (
    df.groupby("event_code")
    .agg(
        plate_count=("plate_clean", "size"),
        minimum_price=("hammer_price_gbp", "min"),
        median_price=("hammer_price_gbp", "median"),
        mean_price=("hammer_price_gbp", "mean"),
        percentile_90_price=(
            "hammer_price_gbp",
            lambda values: values.quantile(0.90)
        ),
        maximum_price=("hammer_price_gbp", "max"),
    )
    .reset_index()
)

event_price_summary = add_pricing_diagnostics(
    event_price_summary
)

event_price_summary


Out[0]: 
  event_code  plate_count  minimum_price  median_price  mean_price  \
0       B270         1966         200.00      1,485.00    2,415.45   
1       B271         1969         200.00      1,510.00    2,503.05   
2       B272         1983         200.00      1,510.00    2,541.98   
3       B273         1983         200.00      1,510.00    2,521.29   
4       B274         1981         200.00      1,440.00    2,462.47   
5       B275         1970         200.00      1,510.00    2,456.28   
6       B276         1972         200.00      1,370.00    2,463.58   
7       B277         1977         200.00      1,520.00    2,523.96   
8       B278         1981         200.00      1,500.00    2,572.35   

   percentile_90_price  maximum_price  median_lift_vs_overall  \
0             5,640.00      33,670.00                    0.98   
1             5,510.00      70,000.00                    1.00   
2             5,910.00      91,020.00                    1.00   
3             5,518.00      56

## 4. Event-Based Validation Structure

The feature table preserves `validation_group` as the original auction-event grouping. A separate `model_split` field is created for the later modelling notebook so the meaning of each column stays stable.

In [11]:
validation_group_map = {
    "B270": "train",
    "B271": "train",
    "B272": "train",
    "B273": "train",
    "B274": "train",
    "B275": "train",
    "B276": "train",
    "B277": "validation",
    "B278": "test",
}

df["model_split"] = (
    df["event_code"]
    .map(validation_group_map)
)

validation_summary = (
    df.groupby(
        ["model_split", "event_code"]
    )
    .agg(
        plate_count=("plate_clean", "size"),
        median_price=("hammer_price_gbp", "median"),
    )
    .reset_index()
    .sort_values(["model_split", "event_code"])
)

validation_summary = add_pricing_diagnostics(
    validation_summary
)

validation_summary


Out[0]: 
  model_split event_code  plate_count  median_price  median_lift_vs_overall  \
0        test       B278         1981      1,500.00                    0.99   
1       train       B270         1966      1,485.00                    0.98   
2       train       B271         1969      1,510.00                    1.00   
3       train       B272         1983      1,510.00                    1.00   
4       train       B273         1983      1,510.00                    1.00   
5       train       B274         1981      1,440.00                    0.95   
6       train       B275         1970      1,510.00                    1.00   
7       train       B276         1972      1,370.00                    0.91   
8  validation       B277         1977      1,520.00                    1.01   

    sample_size_flag  
0  sufficient_sample  
1  sufficient_sample  
2  sufficient_sample  
3  sufficient_sample  
4  sufficient_sample  
5  sufficient_sample  
6  sufficient_sample  
7  sufficient_sa

In [12]:
validation_event_check = (
    df[
        [
            "validation_group",
            "model_split",
            "event_code",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "model_split",
            "event_code",
        ]
    )
)

validation_event_check


Out[0]: 
      validation_group model_split event_code
15801             B278        test       B278
0                 B270       train       B270
1966              B271       train       B271
3935              B272       train       B272
5918              B273       train       B273
7901              B274       train       B274
9882              B275       train       B275
11852             B276       train       B276
13824             B277  validation       B277


In [13]:
assert df.loc[
    df["event_code"].isin(
        [
            "B270",
            "B271",
            "B272",
            "B273",
            "B274",
            "B275",
            "B276",
        ]
    ),
    "model_split",
].eq("train").all()

assert df.loc[
    df["event_code"].eq("B277"),
    "model_split",
].eq("validation").all()

assert df.loc[
    df["event_code"].eq("B278"),
    "model_split",
].eq("test").all()

assert df["validation_group"].eq(df["event_code"]).all(), (
    "validation_group should remain the original event-level grouping."
)

print("Event-based model splits are correctly assigned.")


Event-based model splits are correctly assigned.


## 5. Plate Length and Price

Shorter registrations are expected to be scarcer and potentially more valuable. Median and upper-tail prices are compared by plate length.

In [14]:
plate_length_summary = (
    df.groupby("plate_length")
    .agg(
        plate_count=("plate_clean", "size"),
        median_price=("hammer_price_gbp", "median"),
        mean_price=("hammer_price_gbp", "mean"),
        percentile_90_price=(
            "hammer_price_gbp",
            lambda values: values.quantile(0.90)
        ),
        maximum_price=("hammer_price_gbp", "max"),
    )
    .reset_index()
    .sort_values("plate_length")
)

plate_length_summary = add_pricing_diagnostics(
    plate_length_summary
)

plate_length_summary


Out[0]: 
   plate_length  plate_count  median_price  mean_price  percentile_90_price  \
0             3            8     66,550.00   67,518.75            93,603.00   
1             4         1258      6,510.00    7,624.41            13,000.00   
2             5         4564      2,710.00    3,351.11             6,210.00   
3             6         6715      1,110.00    1,753.27             3,910.00   
4             7         5237        990.00    1,370.88             3,004.00   

   maximum_price  median_lift_vs_overall   sample_size_flag  
0     102,010.00                   44.07       small_sample  
1      91,020.00                    4.31  sufficient_sample  
2      56,040.00                    1.79  sufficient_sample  
3      80,810.00                    0.74  sufficient_sample  
4      31,010.00                    0.66  sufficient_sample  


In [15]:
plot_data = (
    plate_length_summary
    .set_index("plate_length")["median_price"]
)

ax = plot_data.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_title("Median Hammer Price by Plate Length")
ax.set_xlabel("Plate Length")
ax.set_ylabel("Median Hammer Price (£)")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

<ipython-input-1-f393df5b2099>:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
short_plate_comparison = (
    df.groupby(
        "is_short_plate_len_4_or_less"
    )
    .agg(
        plate_count=("plate_clean", "size"),
        median_price=("hammer_price_gbp", "median"),
        mean_price=("hammer_price_gbp", "mean"),
        percentile_90_price=(
            "hammer_price_gbp",
            lambda values: values.quantile(0.90)
        ),
    )
    .reset_index()
)

short_plate_comparison[
    "plate_group"
] = short_plate_comparison[
    "is_short_plate_len_4_or_less"
].map(
    {
        0: "Length above 4",
        1: "Length 4 or below",
    }
)

short_plate_comparison = add_pricing_diagnostics(
    short_plate_comparison
)

short_plate_comparison


Out[0]: 
   is_short_plate_len_4_or_less  plate_count  median_price  mean_price  \
0                             0        16516      1,300.00    2,073.56   
1                             1         1266      6,515.00    8,002.89   

   percentile_90_price        plate_group  median_lift_vs_overall  \
0             4,610.00     Length above 4                    0.86   
1            13,025.00  Length 4 or below                    4.31   

    sample_size_flag  
0  sufficient_sample  
1  sufficient_sample  


## 6. Numeric Rarity and Price

Low and distinctive numeric components may increase scarcity, memorability, and perceived value. Price distributions are compared across numeric-value groups.

In [17]:
numeric_bins = [
    -np.inf,
    9,
    99,
    999,
    9_999,
    np.inf,
]

numeric_labels = [
    "0–9",
    "10–99",
    "100–999",
    "1,000–9,999",
    "10,000+",
]

df["numeric_value_band"] = pd.cut(
    df["numeric_value"],
    bins=numeric_bins,
    labels=numeric_labels,
)

In [18]:
numeric_band_summary = (
    df.groupby(
        "numeric_value_band",
        observed=True,
    )
    .agg(
        plate_count=("plate_clean", "size"),
        median_price=("hammer_price_gbp", "median"),
        mean_price=("hammer_price_gbp", "mean"),
        percentile_90_price=(
            "hammer_price_gbp",
            lambda values: values.quantile(0.90),
        ),
        maximum_price=("hammer_price_gbp", "max"),
    )
    .reset_index()
)

numeric_band_summary = add_pricing_diagnostics(
    numeric_band_summary
)

numeric_band_summary


Out[0]: 
  numeric_value_band  plate_count  median_price  mean_price  \
0                0–9         2862      2,735.00    3,810.06   
1              10–99         7964      1,360.00    2,274.59   
2            100–999         6343      1,110.00    2,108.28   
3        1,000–9,999          613      2,610.00    3,240.72   

   percentile_90_price  maximum_price  median_lift_vs_overall  \
0             8,010.00      90,000.00                    1.81   
1             5,287.00     102,010.00                    0.90   
2             5,010.00      91,020.00                    0.74   
3             5,140.00      33,670.00                    1.73   

    sample_size_flag  
0  sufficient_sample  
1  sufficient_sample  
2  sufficient_sample  
3  sufficient_sample  


In [19]:
plot_data = numeric_band_summary.set_index(
    "numeric_value_band"
)["median_price"]

ax = plot_data.plot(
    kind="bar",
    figsize=(10, 5),
)

ax.set_title("Median Hammer Price by Numeric Value Band")
ax.set_xlabel("Numeric Value Band")
ax.set_ylabel("Median Hammer Price (£)")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

<ipython-input-1-544b88a27fb3>:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
low_number_comparison = (
    df.groupby("is_number_below_100")
    .agg(
        plate_count=("plate_clean", "size"),
        median_price=("hammer_price_gbp", "median"),
        mean_price=("hammer_price_gbp", "mean"),
        percentile_90_price=(
            "hammer_price_gbp",
            lambda values: values.quantile(0.90),
        ),
    )
    .reset_index()
)

low_number_comparison[
    "number_group"
] = low_number_comparison[
    "is_number_below_100"
].map(
    {
        0: "Number 100 or above",
        1: "Number below 100",
    }
)

low_number_comparison = add_pricing_diagnostics(
    low_number_comparison
)

low_number_comparison


Out[0]: 
   is_number_below_100  plate_count  median_price  mean_price  \
0                    0         6956      1,210.00    2,208.08   
1                    1        10826      1,610.00    2,680.52   

   percentile_90_price         number_group  median_lift_vs_overall  \
0             5,010.00  Number 100 or above                    0.80   
1             6,010.00     Number below 100                    1.07   

    sample_size_flag  
0  sufficient_sample  
1  sufficient_sample  


## 7. Repetition and Price

Repeated digits and letters are assessed as possible memorability and collectibility signals.

In [21]:
repeated_digit_summary = (
    df.groupby("has_repeated_digit")
    .agg(
        plate_count=("plate_clean", "size"),
        median_price=("hammer_price_gbp", "median"),
        mean_price=("hammer_price_gbp", "mean"),
        percentile_90_price=(
            "hammer_price_gbp",
            lambda values: values.quantile(0.90),
        ),
    )
    .reset_index()
)

repeated_digit_summary[
    "digit_repetition_group"
] = repeated_digit_summary[
    "has_repeated_digit"
].map(
    {
        0: "No repeated digit",
        1: "Repeated digit",
    }
)

repeated_digit_summary = add_pricing_diagnostics(
    repeated_digit_summary
)

repeated_digit_summary


Out[0]: 
   has_repeated_digit  plate_count  median_price  mean_price  \
0                   0        12079      1,510.00    2,604.43   
1                   1         5703      1,390.00    2,265.43   

   percentile_90_price digit_repetition_group  median_lift_vs_overall  \
0             6,002.00      No repeated digit                    1.00   
1             5,010.00         Repeated digit                    0.92   

    sample_size_flag  
0  sufficient_sample  
1  sufficient_sample  


In [22]:
repeated_letter_summary = (
    df.groupby("has_repeated_letter")
    .agg(
        plate_count=("plate_clean", "size"),
        median_price=("hammer_price_gbp", "median"),
        mean_price=("hammer_price_gbp", "mean"),
        percentile_90_price=(
            "hammer_price_gbp",
            lambda values: values.quantile(0.90),
        ),
    )
    .reset_index()
)

repeated_letter_summary[
    "letter_repetition_group"
] = repeated_letter_summary[
    "has_repeated_letter"
].map(
    {
        0: "No repeated letter",
        1: "Repeated letter",
    }
)

repeated_letter_summary = add_pricing_diagnostics(
    repeated_letter_summary
)

repeated_letter_summary


Out[0]: 
   has_repeated_letter  plate_count  median_price  mean_price  \
0                    0        13987      1,510.00    2,575.19   
1                    1         3795      1,260.00    2,202.74   

   percentile_90_price letter_repetition_group  median_lift_vs_overall  \
0             5,940.00      No repeated letter                    1.00   
1             4,910.00         Repeated letter                    0.83   

    sample_size_flag  
0  sufficient_sample  
1  sufficient_sample  


In [23]:
same_digit_summary = (
    df.groupby("all_digits_same")
    .agg(
        plate_count=("plate_clean", "size"),
        median_price=("hammer_price_gbp", "median"),
        mean_price=("hammer_price_gbp", "mean"),
        percentile_90_price=(
            "hammer_price_gbp",
            lambda values: values.quantile(0.90),
        ),
    )
    .reset_index()
)

same_digit_summary[
    "same_digit_group"
] = same_digit_summary[
    "all_digits_same"
].map(
    {
        0: "Digits not all identical",
        1: "All digits identical",
    }
)

same_digit_summary = add_pricing_diagnostics(
    same_digit_summary
)

same_digit_summary


Out[0]: 
   all_digits_same  plate_count  median_price  mean_price  \
0                0        15376      1,510.00    2,514.83   
1                1         2406      1,500.00    2,373.50   

   percentile_90_price          same_digit_group  median_lift_vs_overall  \
0             5,760.00  Digits not all identical                    1.00   
1             5,380.00      All digits identical                    0.99   

    sample_size_flag  
0  sufficient_sample  
1  sufficient_sample  


## 8. Plate Format and Price

Broad plate-format groups are compared to determine whether the ordering of letters and digits is associated with different price levels.

In [24]:
format_price_summary = (
    df.groupby("plate_format_group")
    .agg(
        plate_count=("plate_clean", "size"),
        median_price=("hammer_price_gbp", "median"),
        mean_price=("hammer_price_gbp", "mean"),
        percentile_90_price=(
            "hammer_price_gbp",
            lambda values: values.quantile(0.90),
        ),
        maximum_price=("hammer_price_gbp", "max"),
    )
    .reset_index()
    .sort_values(
        "median_price",
        ascending=False,
    )
)

format_price_summary = add_pricing_diagnostics(
    format_price_summary
)

format_price_summary


Out[0]: 
       plate_format_group  plate_count  median_price  mean_price  \
0     digits_then_letters         4580      3,910.00    4,873.42   
2     letters_then_digits         1188      1,200.00    2,334.60   
1  letters_digits_letters        12014      1,010.00    1,605.20   

   percentile_90_price  maximum_price  median_lift_vs_overall  \
0             9,001.00     102,010.00                    2.59   
2             5,000.00      26,010.00                    0.79   
1             3,510.00      80,810.00                    0.67   

    sample_size_flag  
0  sufficient_sample  
2  sufficient_sample  
1  sufficient_sample  


In [25]:
plot_data = (
    format_price_summary
    .set_index("plate_format_group")["median_price"]
    .sort_values(ascending=False)
)

ax = plot_data.plot(
    kind="bar",
    figsize=(11, 5),
)

ax.set_title("Median Hammer Price by Plate Format Group")
ax.set_xlabel("Plate Format Group")
ax.set_ylabel("Median Hammer Price (£)")
ax.grid(axis="y", alpha=0.3)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

<ipython-input-1-dfe7d9dba1b9>:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [26]:
pattern_price_summary = (
    df.groupby("plate_pattern")
    .agg(
        plate_count=("plate_clean", "size"),
        median_price=("hammer_price_gbp", "median"),
        mean_price=("hammer_price_gbp", "mean"),
        percentile_90_price=(
            "hammer_price_gbp",
            lambda values: values.quantile(0.90),
        ),
    )
    .reset_index()
)

pattern_price_summary = (
    pattern_price_summary[
        pattern_price_summary["plate_count"] >= minimum_reliable_group_size
    ]
    .sort_values(
        "median_price",
        ascending=False,
    )
)

pattern_price_summary = add_pricing_diagnostics(
    pattern_price_summary
)

pattern_price_summary


Out[0]: 
   plate_pattern  plate_count  median_price  mean_price  percentile_90_price  \
2           DDDL          123     10,620.00   12,559.59            18,330.00   
6           DDLL          309      7,010.00    7,837.25            10,502.00   
9           DLLL          676      5,510.00    6,812.03            12,070.00   
0          DDDDL           53      5,140.00    7,422.64            12,028.00   
3          DDDLL          629      4,640.00    5,240.37             8,024.00   
17          LLLD          144      4,210.00    6,790.56            15,784.00   
7          DDLLL         1436      3,115.00    3,865.47             6,700.00   
1         DDDDLL          455      2,810.00    3,270.59             4,964.00   
4         DDDLLL          891      1,990.00    2,783.52             5,520.00   
23         LLLDL         1886      1,895.00    2,580.19             5,010.00   
16       LLDDLLL          825      1,310.00    1,895.05             3,578.00   
18         LLLDD          542  

## 9. Combined Scarcity Signals

Interactions between short plate length and low numeric values are examined because premium value may arise from combinations of characteristics rather than from a single feature.

In [27]:
df["short_low_number_group"] = np.select(
    [
        (
            df["is_short_plate_len_4_or_less"].eq(1)
            & df["is_number_below_100"].eq(1)
        ),
        (
            df["is_short_plate_len_4_or_less"].eq(1)
            & df["is_number_below_100"].eq(0)
        ),
        (
            df["is_short_plate_len_4_or_less"].eq(0)
            & df["is_number_below_100"].eq(1)
        ),
    ],
    [
        "Short + number below 100",
        "Short + number 100 or above",
        "Long + number below 100",
    ],
    default="Long + number 100 or above",
)

In [28]:
short_low_number_summary = (
    df.groupby("short_low_number_group")
    .agg(
        plate_count=("plate_clean", "size"),
        median_price=("hammer_price_gbp", "median"),
        mean_price=("hammer_price_gbp", "mean"),
        percentile_90_price=(
            "hammer_price_gbp",
            lambda values: values.quantile(0.90),
        ),
        maximum_price=("hammer_price_gbp", "max"),
    )
    .reset_index()
    .sort_values(
        "median_price",
        ascending=False,
    )
)

short_low_number_summary = add_pricing_diagnostics(
    short_low_number_summary
)

short_low_number_summary


Out[0]: 
        short_low_number_group  plate_count  median_price  mean_price  \
2  Short + number 100 or above          123     10,620.00   12,559.59   
3     Short + number below 100         1143      6,130.00    7,512.54   
1      Long + number below 100         9683      1,410.00    2,110.13   
0   Long + number 100 or above         6833      1,200.00    2,021.74   

   percentile_90_price  maximum_price  median_lift_vs_overall  \
2            18,330.00      91,020.00                    7.03   
3            12,010.00     102,010.00                    4.06   
1             4,520.00      80,810.00                    0.93   
0             4,708.00      33,670.00                    0.79   

    sample_size_flag  
2  sufficient_sample  
3  sufficient_sample  
1  sufficient_sample  
0  sufficient_sample  


In [29]:
df["short_same_digit_group"] = np.select(
    [
        (
            df["is_short_plate_len_4_or_less"].eq(1)
            & df["all_digits_same"].eq(1)
        ),
        (
            df["is_short_plate_len_4_or_less"].eq(1)
            & df["all_digits_same"].eq(0)
        ),
        (
            df["is_short_plate_len_4_or_less"].eq(0)
            & df["all_digits_same"].eq(1)
        ),
    ],
    [
        "Short + identical digits",
        "Short + non-identical digits",
        "Long + identical digits",
    ],
    default="Long + non-identical digits",
)

In [30]:
short_same_digit_summary = (
    df.groupby("short_same_digit_group")
    .agg(
        plate_count=("plate_clean", "size"),
        median_price=("hammer_price_gbp", "median"),
        mean_price=("hammer_price_gbp", "mean"),
        percentile_90_price=(
            "hammer_price_gbp",
            lambda values: values.quantile(0.90),
        ),
    )
    .reset_index()
    .sort_values(
        "median_price",
        ascending=False,
    )
)

short_same_digit_summary = add_pricing_diagnostics(
    short_same_digit_summary
)

short_same_digit_summary


Out[0]: 
         short_same_digit_group  plate_count  median_price  mean_price  \
2      Short + identical digits           45      8,000.00    8,161.11   
3  Short + non-identical digits         1221      6,510.00    7,997.06   
0       Long + identical digits         2361      1,430.00    2,263.19   
1   Long + non-identical digits        14155      1,260.00    2,041.94   

   percentile_90_price  median_lift_vs_overall   sample_size_flag  
2            10,442.00                    5.30  sufficient_sample  
3            13,110.00                    4.31  sufficient_sample  
0             5,010.00                    0.95  sufficient_sample  
1             4,510.00                    0.83  sufficient_sample  


In [31]:
premium_rate_summary = pd.DataFrame(
    {
        "feature": [
            "Short plate, length <= 4",
            "Number below 100",
            "Repeated digit",
            "Repeated letter",
            "All digits identical",
        ],
        "plate_count": [
            int(df["is_short_plate_len_4_or_less"].sum()),
            int(df["is_number_below_100"].sum()),
            int(df["has_repeated_digit"].sum()),
            int(df["has_repeated_letter"].sum()),
            int(df["all_digits_same"].sum()),
        ],
        "premium_top_10_rate_pct": [
            (
                df.loc[
                    df["is_short_plate_len_4_or_less"].eq(1),
                    "is_premium_top_10pct",
                ].mean()
                * 100
            ),
            (
                df.loc[
                    df["is_number_below_100"].eq(1),
                    "is_premium_top_10pct",
                ].mean()
                * 100
            ),
            (
                df.loc[
                    df["has_repeated_digit"].eq(1),
                    "is_premium_top_10pct",
                ].mean()
                * 100
            ),
            (
                df.loc[
                    df["has_repeated_letter"].eq(1),
                    "is_premium_top_10pct",
                ].mean()
                * 100
            ),
            (
                df.loc[
                    df["all_digits_same"].eq(1),
                    "is_premium_top_10pct",
                ].mean()
                * 100
            ),
        ],
    }
)

premium_rate_summary = add_pricing_diagnostics(
    premium_rate_summary,
    median_column="median_price",
)

premium_rate_summary.sort_values(
    "premium_top_10_rate_pct",
    ascending=False,
)


Out[0]: 
                    feature  plate_count  premium_top_10_rate_pct  \
0  Short plate, length <= 4         1266                    62.56   
1          Number below 100        10826                    11.47   
4      All digits identical         2406                     8.94   
2            Repeated digit         5703                     7.77   
3           Repeated letter         3795                     7.14   

   premium_top_10_lift_vs_baseline   sample_size_flag  
0                             6.25  sufficient_sample  
1                             1.15  sufficient_sample  
4                             0.89  sufficient_sample  
2                             0.78  sufficient_sample  
3                             0.71  sufficient_sample  


## 10. Commercial Signal Summary

The strongest observed pricing signals are consolidated into a stakeholder-friendly summary. These results describe association rather than causation.

In [32]:
commercial_signal_summary = pd.DataFrame(
    {
        "signal": [
            "Short plate (length <= 4)",
            "Number below 100",
            "Repeated digit",
            "Repeated letter",
            "All digits identical",
        ],
        "plate_count": [
            int(df["is_short_plate_len_4_or_less"].sum()),
            int(df["is_number_below_100"].sum()),
            int(df["has_repeated_digit"].sum()),
            int(df["has_repeated_letter"].sum()),
            int(df["all_digits_same"].sum()),
        ],
        "median_price_gbp": [
            df.loc[
                df["is_short_plate_len_4_or_less"].eq(1),
                "hammer_price_gbp",
            ].median(),
            df.loc[
                df["is_number_below_100"].eq(1),
                "hammer_price_gbp",
            ].median(),
            df.loc[
                df["has_repeated_digit"].eq(1),
                "hammer_price_gbp",
            ].median(),
            df.loc[
                df["has_repeated_letter"].eq(1),
                "hammer_price_gbp",
            ].median(),
            df.loc[
                df["all_digits_same"].eq(1),
                "hammer_price_gbp",
            ].median(),
        ],
        "premium_top_10_rate_pct": [
            df.loc[
                df["is_short_plate_len_4_or_less"].eq(1),
                "is_premium_top_10pct",
            ].mean() * 100,
            df.loc[
                df["is_number_below_100"].eq(1),
                "is_premium_top_10pct",
            ].mean() * 100,
            df.loc[
                df["has_repeated_digit"].eq(1),
                "is_premium_top_10pct",
            ].mean() * 100,
            df.loc[
                df["has_repeated_letter"].eq(1),
                "is_premium_top_10pct",
            ].mean() * 100,
            df.loc[
                df["all_digits_same"].eq(1),
                "is_premium_top_10pct",
            ].mean() * 100,
        ],
    }
)

commercial_signal_summary = add_pricing_diagnostics(
    commercial_signal_summary,
    median_column="median_price_gbp",
)

commercial_signal_summary.sort_values(
    "premium_top_10_rate_pct",
    ascending=False,
)


Out[0]: 
                      signal  plate_count  median_price_gbp  \
0  Short plate (length <= 4)         1266          6,515.00   
1           Number below 100        10826          1,610.00   
4       All digits identical         2406          1,500.00   
2             Repeated digit         5703          1,390.00   
3            Repeated letter         3795          1,260.00   

   premium_top_10_rate_pct  median_lift_vs_overall  \
0                    62.56                    4.31   
1                    11.47                    1.07   
4                     8.94                    0.99   
2                     7.77                    0.92   
3                     7.14                    0.83   

   premium_top_10_lift_vs_baseline   sample_size_flag  
0                             6.25  sufficient_sample  
1                             1.15  sufficient_sample  
4                             0.89  sufficient_sample  
2                             0.78  sufficient_sample  
3      

In [33]:
premium_combination_summary = (
    df.groupby(
        [
            "is_short_plate_len_4_or_less",
            "is_number_below_100",
            "all_digits_same",
        ]
    )
    .agg(
        plate_count=("plate_clean", "size"),
        median_price=("hammer_price_gbp", "median"),
        mean_price=("hammer_price_gbp", "mean"),
        premium_top_10_rate=(
            "is_premium_top_10pct",
            "mean",
        ),
    )
    .reset_index()
)

premium_combination_summary[
    "premium_top_10_rate_pct"
] = (
    premium_combination_summary[
        "premium_top_10_rate"
    ]
    * 100
)

premium_combination_summary = add_pricing_diagnostics(
    premium_combination_summary
)

premium_combination_summary.sort_values(
    [
        "premium_top_10_rate_pct",
        "median_price",
    ],
    ascending=False,
)


Out[0]: 
   is_short_plate_len_4_or_less  is_number_below_100  all_digits_same  \
4                             1                    0                0   
6                             1                    1                1   
5                             1                    1                0   
3                             0                    1                1   
1                             0                    0                1   
0                             0                    0                0   
2                             0                    1                0   

   plate_count  median_price  mean_price  premium_top_10_rate  \
4          123     10,620.00   12,559.59                 1.00   
6           45      8,000.00    8,161.11                 0.93   
5         1098      6,060.00    7,485.96                 0.57   
3         1553      1,490.00    2,273.32                 0.07   
1          808      1,390.00    2,243.73                 0.07   
0         6025  

In [34]:
top_value_plates = (
    df[
        [
            "event_code",
            "lot_number",
            "plate_raw",
            "hammer_price_gbp",
            "plate_length",
            "numeric_value",
            "plate_pattern",
            "plate_format_group",
            "is_number_below_100",
            "all_digits_same",
            "is_premium_top_1pct",
        ]
    ]
    .sort_values(
        "hammer_price_gbp",
        ascending=False,
    )
    .head(25)
)

top_value_plates

Out[0]: 
      event_code  lot_number plate_raw  hammer_price_gbp  plate_length  \
13180       B276        1348      52 O        102,010.00             3   
5232        B272        1308     101 O         91,020.00             4   
8539        B274         646      8 FU         90,000.00             3   
12386       B276         541   ELL 10T         80,810.00             6   
7901        B274           1      95 A         71,010.00             3   
2597        B271         643      3 FU         70,000.00             3   
9219        B274        1332      98 O         63,100.00             3   
13185       B276        1353     9 OAT         62,030.00             4   
7215        B273        1309      54 O         56,510.00             3   
9221        B274        1334    OAS 1S         56,040.00             5   
17589       B278        1805    TON 1S         51,020.00             5   
9833        B274        1951      9 XN         45,010.00             3   
6546        B273         635 

In [35]:
REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

event_price_summary.to_csv(
    REPORTS_DIR / "event_price_summary_2025.csv",
    index=False,
)

validation_summary.to_csv(
    REPORTS_DIR / "model_split_summary_2025.csv",
    index=False,
)

plate_length_summary.to_csv(
    REPORTS_DIR / "plate_length_price_summary_2025.csv",
    index=False,
)

short_plate_comparison.to_csv(
    REPORTS_DIR / "short_plate_comparison_2025.csv",
    index=False,
)

numeric_band_summary.to_csv(
    REPORTS_DIR / "numeric_band_price_summary_2025.csv",
    index=False,
)

low_number_comparison.to_csv(
    REPORTS_DIR / "low_number_comparison_2025.csv",
    index=False,
)

repeated_digit_summary.to_csv(
    REPORTS_DIR / "repeated_digit_summary_2025.csv",
    index=False,
)

repeated_letter_summary.to_csv(
    REPORTS_DIR / "repeated_letter_summary_2025.csv",
    index=False,
)

same_digit_summary.to_csv(
    REPORTS_DIR / "same_digit_summary_2025.csv",
    index=False,
)

format_price_summary.to_csv(
    REPORTS_DIR / "format_price_summary_2025.csv",
    index=False,
)

pattern_price_summary.to_csv(
    REPORTS_DIR / "pattern_price_summary_2025.csv",
    index=False,
)

short_low_number_summary.to_csv(
    REPORTS_DIR / "short_low_number_summary_2025.csv",
    index=False,
)

short_same_digit_summary.to_csv(
    REPORTS_DIR / "short_same_digit_summary_2025.csv",
    index=False,
)

premium_rate_summary.to_csv(
    REPORTS_DIR / "premium_rate_summary_2025.csv",
    index=False,
)

commercial_signal_summary.to_csv(
    REPORTS_DIR / "commercial_signal_summary_2025.csv",
    index=False,
)

premium_combination_summary.to_csv(
    REPORTS_DIR / "premium_combination_summary_2025.csv",
    index=False,
)

top_value_plates.to_csv(
    REPORTS_DIR / "top_value_plates_2025.csv",
    index=False,
)

print("Exploratory pricing reports saved successfully.")


Exploratory pricing reports saved successfully.


## 11. Exploratory Analysis Conclusion

The exploratory analysis identified several commercially meaningful pricing signals:

- Plate length is the strongest observed structural signal in this dataset.
- Registrations with four or fewer characters have substantially higher median and upper-tail prices.
- Low numeric values are associated with higher prices, although their effect is weaker than plate length on its own.
- Repeated digits and letters are not independently reliable premium indicators.
- Repetition becomes more meaningful when combined with short plate length and other scarcity signals.
- Plate format and detailed letter-digit structure are strongly associated with price differences.
- Auction-event median prices remain comparatively stable across the 2025 events.
- Premium value appears to arise from combinations of scarcity, brevity, numeric structure, and format rather than from one isolated characteristic.

Interpretation caveats:

- Very short plates show a strong price signal, but the shortest groups have small sample sizes.
- Reported lift metrics describe association relative to the 2025 auction baseline, not causal effect.
- Small-sample flags should be reviewed before using a group-level result in business claims.
- `model_split` is prepared for later modelling, while `validation_group` remains the original event-level grouping.

These findings will guide feature selection and interaction design in the valuation modelling stage. Premium labels derived from hammer price will be treated as targets or reporting fields and will not be used as valuation-model predictors.